In [ ]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D6 — Microsoft FY24 Q1 Press Release
# ============================================================

!pip install pymupdf -q

from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D6"
DOCUMENT_NAME = "Microsoft FY24 Q1 Press Release"
EXPECTED_PAGE_COUNT = 10
EXPECTED_REFERENCE_RECORD_COUNT = 147

EXPECTED_CATEGORY_COUNTS = {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
}

REFERENCE_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit",
    "Reporting Period",
    "Source Location"
]

OUTPUT_DIR = Path("outputs_D6_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Expected records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Upload source PDF
# ============================================================

print("Upload the D6 Microsoft FY24 Q1 press-release PDF.")

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError("Upload exactly one PDF file.")

SOURCE_PATH = pdf_files[0]

print("Loaded:", SOURCE_PATH.name)


In [ ]:
# ============================================================
# 3. File hash and PDF text extraction
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)

document = fitz.open(SOURCE_PATH)

page_texts = []
for page_index, page in enumerate(document):
    page_texts.append({
        "Page Number": page_index + 1,
        "Text": page.get_text("text")
    })

full_text = "\n".join(page["Text"] for page in page_texts)

print("SHA-256:", SOURCE_SHA256)
print("Pages:", len(document))
print("Characters:", len(full_text))

if len(document) != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"but found {len(document)}."
    )

if not full_text.strip():
    raise ValueError(
        "No machine-readable text was extracted from D6."
    )

In [ ]:
# ============================================================
# 4. Document characterisation
# ============================================================

page_characterisation_rows = []

for page in page_texts:
    text = page["Text"]

    page_characterisation_rows.append({
        "Page Number": page["Page Number"],
        "Character Count": len(text),
        "Word Count": len(text.split()),
        "Numeric Token Count": len(
            re.findall(r"\(?-?\d+(?:,\d{3})*(?:\.\d+)?\)?", text)
        ),
        "Money Token Count": len(
            re.findall(r"\$\s?\d[\d,]*(?:\.\d+)?", text)
        ),
        "Percentage Token Count": len(
            re.findall(r"\(?-?\d+(?:\.\d+)?\)?\s?%", text)
        ),
        "Contains Table": any(
            heading in text
            for heading in [
                "Constant Currency Reconciliation",
                "INCOME STATEMENTS",
                "COMPREHENSIVE INCOME STATEMENTS",
                "BALANCE SHEETS",
                "CASH FLOWS STATEMENTS",
                "SEGMENT REVENUE AND OPERATING INCOME"
            ]
        )
    })

page_characterisation_df = pd.DataFrame(page_characterisation_rows)

numeric_tokens = re.findall(
    r"\(?-?\d+(?:,\d{3})*(?:\.\d+)?\)?",
    full_text
)

word_count = len(
    full_text.split()
)

numeric_token_to_word_ratio = (
    len(numeric_tokens) / word_count
    if word_count
    else 0
)

DOCUMENT_CHARACTERISATION = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": "PDF",
    "page_count": len(document),
    "page_count_valid": len(document) == EXPECTED_PAGE_COUNT,
    "text_extractable": bool(full_text.strip()),
    "ocr_required": not bool(full_text.strip()),
    "total_text_characters": len(full_text),
    "total_text_words": word_count,
    "numeric_token_count": len(numeric_tokens),
    "numeric_token_to_word_ratio": round(
        numeric_token_to_word_ratio,
        3
    ),
    "contains_narrative_highlights": "Business Highlights" in full_text,
    "contains_constant_currency_reconciliation": (
        "Financial Performance Constant Currency Reconciliation" in full_text
    ),
    "contains_income_statements": "INCOME STATEMENTS" in full_text,
    "contains_comprehensive_income_statements": (
        "COMPREHENSIVE INCOME STATEMENTS" in full_text
    ),
    "contains_balance_sheets": "BALANCE SHEETS" in full_text,
    "contains_cash_flows_statements": "CASH FLOWS STATEMENTS" in full_text,
    "contains_segment_table": (
        "SEGMENT REVENUE AND OPERATING INCOME" in full_text
    )
}

display(page_characterisation_df)
print(json.dumps(DOCUMENT_CHARACTERISATION, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 5. Define reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "source_metric_or_financial_statement_line_item",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category":
            "Fixed record category",

        "Statement or Section":
            "Named source statement, table or section",

        "Metric":
            "Source-grounded metric or line-item name",

        "Business Area":
            "Corporate, segment, product or service area",

        "Value 2023":
            "Reported 2023 amount where represented",

        "Value 2022":
            "Reported 2022 amount where represented",

        "GAAP YoY Change":
            "Reported GAAP year-over-year percentage change",

        "Constant Currency Impact":
            (
                "Reported constant-currency impact, preserving "
                "the source unit"
            ),

        "Constant Currency YoY Change":
            (
                "Reported year-over-year constant-currency "
                "percentage change"
            ),

        "Unit":
            "Source-grounded measurement unit",

        "Reporting Period":
            "Explicit source reporting period",

        "Source Location":
            "Exact page and section/table label"
    },

    "branch_reuse":
        (
            "The same fixed reference dataset is reused for "
            "Branches A, B and C."
        )
}


print(
    json.dumps(
        REFERENCE_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 6. Define extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "source_metric_or_financial_statement_line_item",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": ["string", "null"]
        },

        "Statement or Section": {
            "type": ["string", "null"]
        },

        "Metric": {
            "type": ["string", "null"]
        },

        "Business Area": {
            "type": ["string", "null"]
        },

        "Value 2023": {
            "type": ["number", "null"]
        },

        "Value 2022": {
            "type": ["number", "null"]
        },

        "GAAP YoY Change": {
            "type": ["number", "null"]
        },

        "Constant Currency Impact": {
            "type": ["number", "null"]
        },

        "Constant Currency YoY Change": {
            "type": ["number", "null"]
        },

        "Unit": {
            "type": ["string", "null"]
        },

        "Reporting Period": {
            "type": ["string", "null"]
        },

        "Source Location": {
            "type": ["string", "null"]
        }
    },

    "expected_output_structure": {
        "document_id":
            DOCUMENT_ID,

        "records": [
            {
                "Category":
                    "string or null",

                "Statement or Section":
                    "string or null",

                "Metric":
                    "string or null",

                "Business Area":
                    "string or null",

                "Value 2023":
                    "number or null",

                "Value 2022":
                    "number or null",

                "GAAP YoY Change":
                    "number or null",

                "Constant Currency Impact":
                    "number or null",

                "Constant Currency YoY Change":
                    "number or null",

                "Unit":
                    "string or null",

                "Reporting Period":
                    "string or null",

                "Source Location":
                    "string or null"
            }
        ]
    }
}


print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 7. Define fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract the fixed set of quantitative performance observations and
financial-statement line items represented in the Microsoft FY24 Q1
press release.

Include:

1. Quantitative performance highlights on pages 1–2.
2. All rows represented in the three constant-currency reconciliation
   tables on page 3.
3. Primary Income Statement line items on page 6.
4. Primary Comprehensive Income Statement line items on page 7.
5. Primary Balance Sheet line items on page 8.
6. Primary Cash Flow Statement line items on page 9.
7. Primary Segment Revenue and Operating Income line items on page 10.

For every included record, return:

- Category
- Statement or Section
- Metric
- Business Area
- Value 2023
- Value 2022
- GAAP YoY Change
- Constant Currency Impact
- Constant Currency YoY Change
- Unit
- Reporting Period
- Source Location

Rules:

- Extract only information explicitly represented in the source.
- Preserve negative values as negative numbers.
- Preserve reported units and reporting periods.
- Keep repeated metrics as separate observations when they belong to
  different source sections or representations.
- Do not merge narrative and financial-statement observations merely
  because they refer to the same business metric.
- Do not calculate, infer, reconstruct or correct values.
- Exclude publication metadata, contact information, webcast times,
  forward-looking-risk narrative and values appearing only inside
  accounting line-item labels.
- Use null when a schema field is not represented for a particular
  record.
- Return exactly 147 records.
- Return valid JSON using the exact field names defined in the
  extraction schema.
- Do not include explanations before or after the JSON.
"""

print(EXTRACTION_TASK)

In [ ]:
# ============================================================
# 8. Reference-record helper
# ============================================================

reference_records = []


def add_record(
    category,
    statement_or_section,
    metric,
    business_area,
    value_2023=None,
    value_2022=None,
    gaap_yoy_change=None,
    constant_currency_impact=None,
    constant_currency_yoy_change=None,
    unit=None,
    reporting_period=None,
    source_location=None
):
    reference_records.append({
        "Category": category,
        "Statement or Section": statement_or_section,
        "Metric": metric,
        "Business Area": business_area,
        "Value 2023": value_2023,
        "Value 2022": value_2022,
        "GAAP YoY Change": gaap_yoy_change,
        "Constant Currency Impact": constant_currency_impact,
        "Constant Currency YoY Change": constant_currency_yoy_change,
        "Unit": unit,
        "Reporting Period": reporting_period,
        "Source Location": source_location
    })


In [ ]:
# ============================================================
# 9. Narrative performance highlights — pages 1–2
# ============================================================

category = "Narrative performance highlight"
section = "Quarterly results and Business Highlights"
period = "Quarter ended September 30, 2023"

narrative_rows = [
    # metric, business area, value 2023, GAAP growth, CC growth, unit, page
    ("Revenue", "Corporate", 56.5, 13, 12, "USD billion", "Page 1 — Quarterly results"),
    ("Operating income", "Corporate", 26.9, 25, 24, "USD billion", "Page 1 — Quarterly results"),
    ("Net income", "Corporate", 22.3, 27, 26, "USD billion", "Page 1 — Quarterly results"),
    ("Diluted earnings per share", "Corporate", 2.99, 27, 26, "USD per share", "Page 1 — Quarterly results"),
    ("Microsoft Cloud revenue", "Microsoft Cloud", 31.8, 24, 23, "USD billion", "Page 1 — Quarterly results"),
    ("Revenue", "Productivity and Business Processes", 18.6, 13, 12, "USD billion", "Page 1 — Business Highlights"),
    ("Office Commercial products and cloud services revenue", "Office Commercial", None, 15, 14, "percent", "Page 1 — Business Highlights"),
    ("Office 365 Commercial revenue", "Office 365 Commercial", None, 18, 17, "percent", "Page 1 — Business Highlights"),
    ("Office Consumer products and cloud services revenue", "Office Consumer", None, 3, 4, "percent", "Page 1 — Business Highlights"),
    ("Microsoft 365 Consumer subscribers", "Microsoft 365 Consumer", 76.7, None, None, "million subscribers", "Page 1 — Business Highlights"),
    ("LinkedIn revenue", "LinkedIn", None, 8, None, "percent", "Page 1 — Business Highlights"),
    ("Dynamics products and cloud services revenue", "Dynamics", None, 22, 21, "percent", "Page 1 — Business Highlights"),
    ("Dynamics 365 revenue", "Dynamics 365", None, 28, 26, "percent", "Page 1 — Business Highlights"),
    ("Revenue", "Intelligent Cloud", 24.3, 19, None, "USD billion", "Page 1 — Business Highlights"),
    ("Server products and cloud services revenue", "Server products and cloud services", None, 21, None, "percent", "Page 1 — Business Highlights"),
    ("Azure and other cloud services revenue", "Azure and other cloud services", None, 29, 28, "percent", "Page 1 — Business Highlights"),
    ("Revenue", "More Personal Computing", 13.7, 3, 2, "USD billion", "Page 1 — Business Highlights"),
    ("Windows revenue", "Windows", None, 5, None, "percent", "Page 1 — Business Highlights"),
    ("Windows OEM revenue", "Windows OEM", None, 4, None, "percent", "Page 1 — Business Highlights"),
    ("Windows Commercial products and cloud services revenue", "Windows Commercial", None, 8, None, "percent", "Page 1 — Business Highlights"),
    ("Devices revenue", "Devices", None, -22, None, "percent", "Page 1 — Business Highlights"),
    ("Xbox content and services revenue", "Xbox content and services", None, 13, 12, "percent", "Page 1 — Business Highlights"),
    ("Search and news advertising revenue excluding traffic acquisition costs", "Search and news advertising", None, 10, 9, "percent", "Page 1 — Business Highlights"),
    ("Share repurchases and dividends", "Corporate", 9.1, None, None, "USD billion", "Page 2 — Shareholder returns")
]

for metric, area, value_2023, gaap, cc, unit, location in narrative_rows:
    add_record(
        category=category,
        statement_or_section=section,
        metric=metric,
        business_area=area,
        value_2023=value_2023,
        gaap_yoy_change=gaap,
        constant_currency_yoy_change=cc,
        unit=unit,
        reporting_period=period,
        source_location=location
    )

print("Narrative records:", len(reference_records))


In [ ]:
# ============================================================
# 10. Page 3 — constant-currency reconciliation tables
# ============================================================

period = "Three months ended September 30, 2023"

financial_reconciliation = [
    ("Revenue", 56517, 50122, 13, 301, 12, "USD millions; percent"),
    ("Operating income", 26895, 21518, 25, 204, 24, "USD millions; percent"),
    ("Net income", 22291, 17556, 27, 148, 26, "USD millions; percent"),
    ("Diluted earnings per share", 2.99, 2.35, 27, 0.02, 26, "USD per share; percent")
]

for metric, v2023, v2022, gaap, impact, cc, unit in financial_reconciliation:
    add_record(
        "Financial performance reconciliation",
        "Financial Performance Constant Currency Reconciliation",
        metric,
        "Corporate",
        v2023,
        v2022,
        gaap,
        impact,
        cc,
        unit,
        period,
        "Page 3 — Financial Performance Constant Currency Reconciliation"
    )


segment_reconciliation = [
    ("Productivity and Business Processes", 18592, 16465, 13, 79, 12),
    ("Intelligent Cloud", 24259, 20325, 19, 156, 19),
    ("More Personal Computing", 13666, 13332, 3, 66, 2)
]

for area, v2023, v2022, gaap, impact, cc in segment_reconciliation:
    add_record(
        "Segment revenue reconciliation",
        "Segment Revenue Constant Currency Reconciliation",
        "Revenue",
        area,
        v2023,
        v2022,
        gaap,
        impact,
        cc,
        "USD millions; percent",
        period,
        "Page 3 — Segment Revenue Constant Currency Reconciliation"
    )


selected_product_rows = [
    ("Microsoft Cloud", 24, -1, 23),
    ("Office Commercial products and cloud services", 15, -1, 14),
    ("Office 365 Commercial", 18, -1, 17),
    ("Office Consumer products and cloud services", 3, 1, 4),
    ("LinkedIn", 8, 0, 8),
    ("Dynamics products and cloud services", 22, -1, 21),
    ("Dynamics 365", 28, -2, 26),
    ("Server products and cloud services", 21, 0, 21),
    ("Azure and other cloud services", 29, -1, 28),
    ("Windows", 5, 0, 5),
    ("Windows OEM", 4, 0, 4),
    ("Windows Commercial products and cloud services", 8, 0, 8),
    ("Devices", -22, 0, -22),
    ("Xbox content and services", 13, -1, 12),
    ("Search and news advertising excluding traffic acquisition costs", 10, -1, 9)
]

for area, gaap, impact, cc in selected_product_rows:
    add_record(
        "Selected product and service reconciliation",
        "Selected Product and Service Revenue Constant Currency Reconciliation",
        "Revenue",
        area,
        gaap_yoy_change=gaap,
        constant_currency_impact=impact,
        constant_currency_yoy_change=cc,
        unit="percent",
        reporting_period=period,
        source_location=(
            "Page 3 — Selected Product and Service Revenue "
            "Constant Currency Reconciliation"
        )
    )

In [ ]:
# ============================================================
# 11. Page 6 — Income Statements
# ============================================================

income_statement_rows = [
    ("Product revenue", 15535, 15741, "USD millions"),
    ("Service and other revenue", 40982, 34381, "USD millions"),
    ("Total revenue", 56517, 50122, "USD millions"),
    ("Product cost of revenue", 3531, 4302, "USD millions"),
    ("Service and other cost of revenue", 12771, 11150, "USD millions"),
    ("Total cost of revenue", 16302, 15452, "USD millions"),
    ("Gross margin", 40215, 34670, "USD millions"),
    ("Research and development", 6659, 6628, "USD millions"),
    ("Sales and marketing", 5187, 5126, "USD millions"),
    ("General and administrative", 1474, 1398, "USD millions"),
    ("Operating income", 26895, 21518, "USD millions"),
    ("Other income, net", 389, 54, "USD millions"),
    ("Income before income taxes", 27284, 21572, "USD millions"),
    ("Provision for income taxes", 4993, 4016, "USD millions"),
    ("Net income", 22291, 17556, "USD millions"),
    ("Basic earnings per share", 3.00, 2.35, "USD per share"),
    ("Diluted earnings per share", 2.99, 2.35, "USD per share"),
    ("Basic weighted average shares outstanding", 7429, 7457, "million shares"),
    ("Diluted weighted average shares outstanding", 7462, 7485, "million shares")
]

for metric, v2023, v2022, unit in income_statement_rows:
    add_record(
        "Income statement",
        "Income Statements",
        metric,
        "Corporate",
        v2023,
        v2022,
        unit=unit,
        reporting_period="Three months ended September 30",
        source_location="Page 6 — Income Statements"
    )


In [ ]:
# ============================================================
# 12. Page 7 — Comprehensive Income Statements
# ============================================================

comprehensive_income_rows = [
    ("Net income", 22291, 17556),
    ("Net change related to derivatives", 21, 7),
    ("Net change related to investments", -260, -1897),
    ("Translation adjustments and other", -355, -775),
    ("Other comprehensive loss", -594, -2665),
    ("Comprehensive income", 21697, 14891)
]

for metric, v2023, v2022 in comprehensive_income_rows:
    add_record(
        "Comprehensive income statement",
        "Comprehensive Income Statements",
        metric,
        "Corporate",
        v2023,
        v2022,
        unit="USD millions",
        reporting_period="Three months ended September 30",
        source_location="Page 7 — Comprehensive Income Statements"
    )


In [ ]:
# ============================================================
# 13. Page 8 — Balance Sheets
# ============================================================

balance_sheet_rows = [
    ("Cash and cash equivalents", 80452, 34704),
    ("Short-term investments", 63499, 76558),
    ("Total cash, cash equivalents, and short-term investments", 143951, 111262),
    ("Accounts receivable, net", 36953, 48688),
    ("Inventories", 3000, 2500),
    ("Other current assets", 23682, 21807),
    ("Total current assets", 207586, 184257),
    ("Property and equipment, net", 102502, 95641),
    ("Operating lease right-of-use assets", 15435, 14346),
    ("Equity investments", 11423, 9879),
    ("Goodwill", 67790, 67886),
    ("Intangible assets, net", 8895, 9366),
    ("Other long-term assets", 32154, 30601),
    ("Total assets", 445785, 411976),
    ("Accounts payable", 19307, 18095),
    ("Short-term debt", 25808, 0),
    ("Current portion of long-term debt", 3748, 5247),
    ("Accrued compensation", 6990, 11009),
    ("Short-term income taxes", 8035, 4152),
    ("Short-term unearned revenue", 46429, 50901),
    ("Other current liabilities", 14475, 14745),
    ("Total current liabilities", 124792, 104149),
    ("Long-term debt", 41946, 41990),
    ("Long-term income taxes", 22983, 25560),
    ("Long-term unearned revenue", 2759, 2912),
    ("Deferred income taxes", 470, 433),
    ("Operating lease liabilities", 13487, 12728),
    ("Other long-term liabilities", 18634, 17981),
    ("Total liabilities", 225071, 205753),
    ("Common stock and paid-in capital", 95508, 93718),
    ("Retained earnings", 132143, 118848),
    ("Accumulated other comprehensive loss", -6937, -6343),
    ("Total stockholders' equity", 220714, 206223),
    ("Total liabilities and stockholders' equity", 445785, 411976)
]

for metric, v2023, v2022 in balance_sheet_rows:
    add_record(
        "Balance sheet",
        "Balance Sheets",
        metric,
        "Corporate",
        v2023,
        v2022,
        unit="USD millions",
        reporting_period="September 30, 2023 and June 30, 2023",
        source_location="Page 8 — Balance Sheets"
    )


In [ ]:
# ============================================================
# 14. Page 9 — Cash Flows Statements
# ============================================================

cash_flow_rows = [
    ("Net income", 22291, 17556),
    ("Depreciation, amortization, and other", 3921, 2790),
    ("Stock-based compensation expense", 2507, 2192),
    ("Net recognized losses (gains) on investments and derivatives", 14, -22),
    ("Deferred income taxes", -568, -1191),
    ("Accounts receivable", 11034, 11729),
    ("Inventories", -505, -543),
    ("Other current assets", -796, -332),
    ("Other long-term assets", -2013, -666),
    ("Accounts payable", 1214, -1567),
    ("Unearned revenue", -4126, -3322),
    ("Income taxes", 1425, 410),
    ("Other current liabilities", -4106, -4024),
    ("Other long-term liabilities", 291, 188),
    ("Net cash from operations", 30583, 23198),
    ("Proceeds from issuance of debt, maturities of 90 days or less, net", 18692, 0),
    ("Proceeds from issuance of debt", 7073, 0),
    ("Repayments of debt", -1500, -1000),
    ("Common stock issued", 685, 575),
    ("Common stock repurchased", -4831, -5573),
    ("Common stock cash dividends paid", -5051, -4621),
    ("Other, net — financing", -307, -264),
    ("Net cash from (used in) financing", 14761, -10883),
    ("Additions to property and equipment", -9917, -6283),
    ("Acquisition of companies, net of cash acquired, and purchases of intangible and other assets", -1186, -349),
    ("Purchases of investments", -8460, -5013),
    ("Maturities of investments", 15718, 6662),
    ("Sales of investments", 5330, 2711),
    ("Other, net — investing", -982, -860),
    ("Net cash from (used in) investing", 503, -3132),
    ("Effect of foreign exchange rates on cash and cash equivalents", -99, -230),
    ("Net change in cash and cash equivalents", 45748, 8953),
    ("Cash and cash equivalents, beginning of period", 34704, 13931),
    ("Cash and cash equivalents, end of period", 80452, 22884)
]

for metric, v2023, v2022 in cash_flow_rows:
    add_record(
        "Cash flow statement",
        "Cash Flows Statements",
        metric,
        "Corporate",
        v2023,
        v2022,
        unit="USD millions",
        reporting_period="Three months ended September 30",
        source_location="Page 9 — Cash Flows Statements"
    )


In [ ]:
# ============================================================
# 15. Page 10 — Segment Revenue and Operating Income
# ============================================================

segment_table_rows = [
    ("Revenue", "Productivity and Business Processes", 18592, 16465),
    ("Revenue", "Intelligent Cloud", 24259, 20325),
    ("Revenue", "More Personal Computing", 13666, 13332),
    ("Revenue", "Total", 56517, 50122),
    ("Operating income", "Productivity and Business Processes", 9970, 8323),
    ("Operating income", "Intelligent Cloud", 11751, 8978),
    ("Operating income", "More Personal Computing", 5174, 4217),
    ("Operating income", "Total", 26895, 21518)
]

for metric, area, v2023, v2022 in segment_table_rows:
    add_record(
        "Segment revenue and operating income",
        "Segment Revenue and Operating Income",
        metric,
        area,
        v2023,
        v2022,
        unit="USD millions",
        reporting_period="Three months ended September 30",
        source_location="Page 10 — Segment Revenue and Operating Income"
    )

print("Total constructed records:", len(reference_records))

In [ ]:
# ============================================================
# 16. Create reference dataframe
# ============================================================

reference_values_df = pd.DataFrame(
    reference_records,
    columns=REFERENCE_FIELDS
)

display(reference_values_df.head())

print("Shape:", reference_values_df.shape)
print("\nCategory counts:")
print(reference_values_df["Category"].value_counts())

In [ ]:
# ============================================================
# 17. Reference-dataset integrity validation
# ============================================================

schema_valid = (
    reference_values_df.columns.tolist()
    == REFERENCE_FIELDS
)

record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

observed_category_counts = (
    reference_values_df["Category"]
    .value_counts()
    .to_dict()
)

category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

mandatory_fields = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]

mandatory_fields_complete = bool(
    reference_values_df[mandatory_fields]
    .notna()
    .all()
    .all()
)

has_at_least_one_measurement = bool(
    reference_values_df[
        [
            "Value 2023",
            "Value 2022",
            "GAAP YoY Change",
            "Constant Currency Impact",
            "Constant Currency YoY Change"
        ]
    ]
    .notna()
    .any(axis=1)
    .all()
)

duplicate_key_fields = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Source Location"
]

duplicate_key_count = int(
    reference_values_df.duplicated(
        subset=duplicate_key_fields,
        keep=False
    ).sum()
)

negative_values_preserved = any(
    pd.to_numeric(
        reference_values_df[column],
        errors="coerce"
    ).lt(0).any()
    for column in [
        "Value 2023",
        "Value 2022",
        "GAAP YoY Change",
        "Constant Currency Impact",
        "Constant Currency YoY Change"
    ]
)

REFERENCE_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "expected_record_count": EXPECTED_REFERENCE_RECORD_COUNT,
    "observed_record_count": len(reference_values_df),
    "record_count_valid": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "category_counts_valid": category_counts_valid,
    "schema_valid": schema_valid,
    "mandatory_fields_complete": mandatory_fields_complete,
    "all_records_have_measurement": has_at_least_one_measurement,
    "duplicate_key_count": duplicate_key_count,
    "duplicate_keys_absent": duplicate_key_count == 0,
    "negative_values_preserved": negative_values_preserved,
    "reference_integrity_passed": all([
        schema_valid,
        record_count_valid,
        category_counts_valid,
        mandatory_fields_complete,
        has_at_least_one_measurement,
        duplicate_key_count == 0,
        negative_values_preserved
    ])
}

print(json.dumps(REFERENCE_INTEGRITY, indent=2, ensure_ascii=False))

if not REFERENCE_INTEGRITY["reference_integrity_passed"]:
    raise AssertionError(
        "D6 reference-value construction failed one or more integrity checks."
    )

In [ ]:
# ============================================================
# 18. Source-grounding checks
# ============================================================

SOURCE_GROUNDING_CHECKS = {
    "quarterly_results_present": all(
        phrase in full_text
        for phrase in [
            "Revenue was $56.5 billion",
            "Operating income was $26.9 billion",
            "Net income was $22.3 billion",
            "Diluted earnings per share was $2.99"
        ]
    ),
    "microsoft_cloud_revenue_present": (
        "Microsoft Cloud revenue of $31.8 billion" in full_text
    ),
    "reconciliation_tables_present": all(
        phrase in full_text
        for phrase in [
            "Financial Performance Constant Currency Reconciliation",
            "Segment Revenue Constant Currency Reconciliation",
            "Selected Product and Service Revenue Constant Currency Reconciliation"
        ]
    ),
    "financial_statements_present": all(
        phrase in full_text
        for phrase in [
            "INCOME STATEMENTS",
            "COMPREHENSIVE INCOME STATEMENTS",
            "BALANCE SHEETS",
            "CASH FLOWS STATEMENTS",
            "SEGMENT REVENUE AND OPERATING INCOME"
        ]
    ),
    "source_grounding_passed": False
}

SOURCE_GROUNDING_CHECKS["source_grounding_passed"] = all(
    value
    for key, value in SOURCE_GROUNDING_CHECKS.items()
    if key != "source_grounding_passed"
)

print(json.dumps(SOURCE_GROUNDING_CHECKS, indent=2))

if not SOURCE_GROUNDING_CHECKS["source_grounding_passed"]:
    raise AssertionError("One or more required D6 source regions were not detected.")

In [ ]:
# ============================================================
# 19. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Reading Order Quality",

        "Score":
            "Medium",

        "Evidence Source":
            "PDF text extraction + manual document inspection",

        "Justification":
            "The report has generally identifiable reading sequences, "
            "but alternates between narrative highlights and several "
            "dense financial or reconciliation tables that require "
            "structural interpretation."
    },
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Table Structure Integrity",

        "Score":
            "High",

        "Evidence Source":
            "Financial-table inspection + extracted text inspection",

        "Justification":
            "The document contains multiple reconciliation and "
            "financial-statement tables with multi-column headers, "
            "different reporting periods, grouped rows, repeated "
            "metrics, and source-dependent value relationships."
    },
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Section/Header Hierarchy",

        "Score":
            "Medium",

        "Evidence Source":
            "Manual document inspection",

        "Justification":
            "Major report and statement headings are identifiable, "
            "but hierarchy varies between narrative sections, "
            "reconciliation tables, financial statements, and "
            "segment-level reporting."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Sharpness",

        "Score":
            "Low",

        "Evidence Source":
            "Manual visual inspection",

        "Justification":
            "The born-digital PDF is visually clear and financial "
            "text and values are readable."
    },
    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Noise / Degradation",

        "Score":
            "Low",

        "Evidence Source":
            "Manual visual inspection",

        "Justification":
            "No relevant scanning noise, blur, or document "
            "degradation affects readability."
    },
    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "OCR Dependency",

        "Score":
            "Low",

        "Evidence Source":
            "Automated PDF text extraction",

        "Justification":
            "The document contains directly extractable "
            "machine-readable text and does not require OCR."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Terminology Consistency",

        "Score":
            "Low",

        "Evidence Source":
            "Manual financial-content inspection",

        "Justification":
            "Microsoft uses financial, segment, GAAP, and constant-"
            "currency terminology consistently throughout the report."
    },
    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Schema Alignment",

        "Score":
            "Medium",

        "Evidence Source":
            "Reference-schema comparison",

        "Justification":
            "The common extraction schema must represent narrative "
            "performance highlights, reconciliation rows, financial-"
            "statement line items, and segment-level observations "
            "using the same set of fields."
    },
    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Numerical Density",

        "Score":
            "High",

        "Evidence Source":
            "Automated document profiling + manual inspection",

        "Justification":
            "The report contains a high concentration of financial "
            "amounts, percentages, per-share values, subscriber "
            "counts, and multi-period accounting values."
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Required Field Presence",

        "Score":
            "Low",

        "Evidence Source":
            "Reference-value verification",

        "Justification":
            "All information required by the predefined 147-record "
            "extraction scope is represented in the source."
    },
    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Internal Consistency",

        "Score":
            "Medium",

        "Evidence Source":
            "Manual source and reference inspection",

        "Justification":
            "The document is internally coherent, but the same "
            "financial metrics may recur across narrative, "
            "reconciliation, and accounting-statement sections at "
            "different scales or levels of precision."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Format Heterogeneity",

        "Score":
            "High",

        "Evidence Source":
            "Document profiling + manual inspection",

        "Justification":
            "The report combines narrative highlights, business "
            "bullets, three constant-currency reconciliation tables, "
            "and multiple financial-statement structures."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Unit / Label Variability",

        "Score":
            "High",

        "Evidence Source":
            "Reference and source inspection",

        "Justification":
            "The extraction scope includes USD billions, USD millions, "
            "USD per share, percentages, subscriber counts, constant-"
            "currency impacts, different reporting-period conventions, "
            "and negative values represented using accounting notation."
    }
]


indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

In [ ]:
# ============================================================
# 20. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores detected: "
        f"{invalid_scores}"
    )


expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if missing_indicators:
    raise ValueError(
        f"Missing required indicators: "
        f"{missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators detected: "
        f"{unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

In [ ]:
# ============================================================
# 21. Derive dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

display(
    dimension_assessment_df
)

In [ ]:
# ============================================================
# 22. Build structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [ ]:
# ============================================================
# 23. Reference summary and metadata
# ============================================================

REFERENCE_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "reference_record_count": len(reference_values_df),
    "expected_reference_record_count": EXPECTED_REFERENCE_RECORD_COUNT,
    "category_counts": observed_category_counts,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "source_pages_in_scope": [1, 2, 3, 6, 7, 8, 9, 10],
    "excluded_pages_or_regions": [
        "Page 2 non-quantitative outlook, ESG and webcast narrative",
        "Pages 4–5 About Microsoft, forward-looking statements and contacts",
        "Values embedded only in accounting line-item labels"
    ],
    "reference_schema_fields": REFERENCE_FIELDS,
    "reference_integrity_passed": REFERENCE_INTEGRITY[
        "reference_integrity_passed"
    ]
}

REFERENCE_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": "PDF",
    "page_count": len(document),
    "reference_file": "D6_reference_values.csv",
    "expected_record_count": EXPECTED_REFERENCE_RECORD_COUNT,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "reference_construction_method": (
        "Manual document-grounded transcription into a fixed schema"
    ),
    "calculations_applied": False,
    "external_knowledge_used": False,
    "values_inferred": False,
    "negative_values_preserved": True,
    "python_version": sys.version,
    "platform": platform.platform(),
    "reference_values_branch_independent": True,
    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],
}

print(json.dumps(REFERENCE_SUMMARY, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 24. Export outputs
# ============================================================

REFERENCE_CSV_PATH = OUTPUT_DIR / "D6_reference_values.csv"
REFERENCE_JSON_PATH = OUTPUT_DIR / "D6_reference_values.json"
REFERENCE_SCHEMA_PATH = OUTPUT_DIR / "D6_reference_schema.json"
REFERENCE_SUMMARY_PATH = OUTPUT_DIR / "D6_reference_summary.json"
REFERENCE_METADATA_PATH = OUTPUT_DIR / "D6_reference_metadata.json"
REFERENCE_INTEGRITY_PATH = OUTPUT_DIR / "D6_reference_integrity.json"
DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D6_document_characterisation.json"
)
PAGE_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D6_page_characterisation.csv"
)
QUALITY_EVIDENCE_PATH = OUTPUT_DIR / "D6_quality_evidence.json"
EXTRACTION_TASK_PATH = (
    OUTPUT_DIR /
    "D6_extraction_task.txt"
)
INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D6_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D6_dimension_assessment.csv"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR /
    "D6_extraction_schema.json"
)
reference_values_df.to_csv(
    REFERENCE_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

with REFERENCE_JSON_PATH.open("w", encoding="utf-8") as file:
    json.dump(reference_records, file, ensure_ascii=False, indent=2)

json_outputs = [
    (
        EXTRACTION_SCHEMA_PATH,
        EXTRACTION_SCHEMA
    ),
    (
        REFERENCE_SCHEMA_PATH,
        REFERENCE_SCHEMA
    ),
    (
        REFERENCE_SUMMARY_PATH,
        REFERENCE_SUMMARY
    ),
    (
        REFERENCE_METADATA_PATH,
        REFERENCE_METADATA
    ),
    (
        REFERENCE_INTEGRITY_PATH,
        REFERENCE_INTEGRITY
    ),
    (
        DOCUMENT_CHARACTERISATION_PATH,
        DOCUMENT_CHARACTERISATION
    ),
    (
        QUALITY_EVIDENCE_PATH,
        QUALITY_EVIDENCE
    )
]

EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8"
)

for output_path, content in json_outputs:
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(content, file, ensure_ascii=False, indent=2)

page_characterisation_df.to_csv(
    PAGE_CHARACTERISATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("D6 reference outputs exported to:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 25. Final completion checks
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_CSV_PATH,
    REFERENCE_JSON_PATH,
    EXTRACTION_SCHEMA_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    REFERENCE_INTEGRITY_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    PAGE_CHARACTERISATION_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    QUALITY_EVIDENCE_PATH,
    EXTRACTION_TASK_PATH
]

all_outputs_exist = all(path.exists() for path in GENERATED_OUTPUTS)

FINAL_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "reference_records": len(reference_values_df),
    "expected_records": EXPECTED_REFERENCE_RECORD_COUNT,
    "category_counts_valid": category_counts_valid,
    "schema_valid": schema_valid,
    "reference_integrity_passed": REFERENCE_INTEGRITY[
        "reference_integrity_passed"
    ],
    "source_grounding_passed": SOURCE_GROUNDING_CHECKS[
        "source_grounding_passed"
    ],
    "all_outputs_exist": all_outputs_exist,
    "outputs_created": [path.name for path in GENERATED_OUTPUTS]
}

print(json.dumps(FINAL_SUMMARY, indent=2, ensure_ascii=False))

if not all([
    schema_valid,
    record_count_valid,
    category_counts_valid,
    REFERENCE_INTEGRITY["reference_integrity_passed"],
    SOURCE_GROUNDING_CHECKS["source_grounding_passed"],
    all_outputs_exist
]):
    raise AssertionError("D6 reference notebook did not complete successfully.")

print("\nD6 Stage 1 completed successfully.")